<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Session 09
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Batch Processing
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

In [ ]:
from pcamarillor.spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("Batch Processing", "local[*]")
sc = su._spark.sparkContext
sc

In [ ]:
# Create the schema
columns_info = [
    ("date", "date"),
    ("hour", "string"),
    ("passenger_count", "double"),
    ("PU_Borough", "string"),
    ("DO_Borough", "string"),
    ("payment_type", "int"),
    ("trip_count", "int"),
    ("trip_distance_sum", "double"),
    ("duration_sum", "double"),
    ("fare_amount_sum", "double"),
    ("extra_sum", "double"),
    ("mta_tax_sum", "double"),
    ("tip_amount_sum", "double"),
    ("tolls_amount_sum", "double"),
    ("improvement_surcharge_sum", "double"),
    ("congestion_surcharge_sum", "double"),
    ("airport_fee_sum", "double"),
    ("total_amount_sum", "double"),
]

aggregated_trips_schema = SparkUtils.generate_schema(columns_info)
df_nyc_taxi = (
    su._spark.read
    .schema(aggregated_trips_schema)
    #.option("inferSchema", "true") # <--- This option should be avoided in future pipelines
    .option("header", "true")
    .csv("/opt/spark/work-dir/data/nyc_taxi/archive/aggregated_nyc_yellow_taxi_2024.csv")
)

df_nyc_taxi.printSchema()
df_nyc_taxi.show(2)

## Data Cleaning

In [ ]:
n_records = df_nyc_taxi.count()
n_records

In [ ]:
from pyspark.sql.functions import count, when, isnull

df_nyc_taxi.select([count(when(isnull(c), c)).alias(c) for c in df_nyc_taxi.columns]).show()


In [ ]:
df_nyc_taxi.count()

In [ ]:
clean = df_nyc_taxi.dropna()
clean.select([count(when(isnull(c), c)).alias(c) for c in clean.columns]).show()

In [ ]:
clean.count()

In [ ]:
clean_fillna = df_nyc_taxi.fillna({
    'passenger_count': 0
})
clean_fillna.count()

In [ ]:
clean_fillna.select([count(when(isnull(c), c)).alias(c) for c in clean_fillna.columns]).show()

# Basic Transformations

In [ ]:
from pyspark.sql.functions import col, lit, when

## Select

In [ ]:
# Pick a subset of columns to work with
trip_summary_df = df_nyc_taxi.select(
    "date", "hour", "PU_Borough", "DO_Borough", "trip_count", "total_amount_sum"
)
trip_summary_df.show(5)

# select() with col() to reference columns explicitly (useful when combining
# with expressions or disambiguating columns after a join)
fare_df = df_nyc_taxi.select(
    col("date"),
    col("PU_Borough"),
    col("fare_amount_sum"),
    col("tip_amount_sum")
)
fare_df.show(5)

## withColumn + lit()

In [ ]:
# --- withColumn() + lit() ---
# Add a constant/literal column, e.g. tagging the source of this batch
tagged_df = df_nyc_taxi.withColumn("data_source", lit("nyc_tlc_2024"))
tagged_df.select("date", "PU_Borough", "data_source").show(5)

# Derive a new numeric column: average fare per trip
avg_fare_df = df_nyc_taxi.withColumn(
    "avg_fare_per_trip",
    col("fare_amount_sum") / col("trip_count")
)
avg_fare_df.select("date", "hour", "trip_count", "fare_amount_sum", "avg_fare_per_trip").show(5)


## When

In [ ]:
# --- when() ---
# Recode payment_type (1 = credit card, 2 = cash, per TLC docs) into a readable label
payment_label_df = df_nyc_taxi.withColumn(
    "payment_label",
    when(col("payment_type") == 1, lit("Credit Card"))
    .when(col("payment_type") == 2, lit("Cash"))
    .otherwise(lit("Other"))
)
payment_label_df.select("payment_type", "payment_label").distinct().show()

# Flag trips that crossed a bridge/tunnel or airport, based on summed surcharges
flagged_df = df_nyc_taxi.withColumn(
    "trip_type",
    when(col("airport_fee_sum") > 0, lit("Airport"))
    .when(col("tolls_amount_sum") > 0, lit("Toll Road"))
    .otherwise(lit("Standard"))
)
flagged_df.select("date", "hour", "PU_Borough", "DO_Borough", "trip_type").show(10)

## Consolidated example

In [ ]:
enriched_df = (
    df_nyc_taxi
    .withColumn(
        "borough_match",
        when(col("PU_Borough") == col("DO_Borough"), lit("Same Borough"))
        .otherwise(lit("Cross Borough"))
    )
    .select(
        col("date"),
        col("hour"),
        col("PU_Borough"),
        col("DO_Borough"),
        col("borough_match"),
        col("trip_count"),
        col("total_amount_sum")
    )
)
enriched_df.show(10)

In [ ]:
sc.stop()